# Feature Set Experiments

ADD section


In [1]:
import pandas as pd
import numpy as np

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

In [2]:
df = pd.read_csv("../data/processed/f1_full_merged_dataset.csv", low_memory=False)

df.head()

,resultId,raceId,driverId,constructorId,number,grid,position,positionText,positionOrder,points,...,sprint_points,podium,podium_place,driver_points_before_race,driver_standing_before_race,driver_wins_before_race,constructor_points_before_race,constructor_standing_before_race,constructor_wins_before_race,driver_name
0,371,36,1,1,2,4,3,3,3,6.0,...,NaN,1,3.0,0.0,99.0,0.0,0.0,99.0,0.0,Lewis Hamilton
1,392,37,1,1,2,4,2,2,2,8.0,...,NaN,1,2.0,6.0,3.0,0.0,14.0,11.0,0.0,Lewis Hamilton
2,414,38,1,1,2,2,2,2,2,8.0,...,NaN,1,2.0,14.0,3.0,0.0,32.0,11.0,1.0,Lewis Hamilton
3,436,39,1,1,2,4,2,2,2,8.0,...,NaN,1,2.0,22.0,3.0,0.0,44.0,11.0,1.0,Lewis Hamilton
4,458,40,1,1,2,2,2,2,2,8.0,...,NaN,1,2.0,30.0,1.0,0.0,58.0,11.0,1.0,Lewis Hamilton


In [3]:
modeling_data = df[[
    "raceId",
    "driverId",
    "driver_name",
    "constructorId",
    "name_constructor",
    "circuitId",
    "year",
    "round",
    "grid",
    "qualifying_position",
    "driver_points_before_race",
    "driver_standing_before_race",
    "driver_wins_before_race",
    "constructor_points_before_race",
    "constructor_standing_before_race",
    "constructor_wins_before_race",
    "podium"
]].copy()

modeling_data.head()


,raceId,driverId,driver_name,constructorId,name_constructor,circuitId,year,round,grid,qualifying_position,driver_points_before_race,driver_standing_before_race,driver_wins_before_race,constructor_points_before_race,constructor_standing_before_race,constructor_wins_before_race,podium
0,36,1,Lewis Hamilton,1,McLaren,1,2007,1,4,4.0,0.0,99.0,0.0,0.0,99.0,0.0,1
1,37,1,Lewis Hamilton,1,McLaren,2,2007,2,4,4.0,6.0,3.0,0.0,14.0,11.0,0.0,1
2,38,1,Lewis Hamilton,1,McLaren,3,2007,3,2,2.0,14.0,3.0,0.0,32.0,11.0,1.0,1
3,39,1,Lewis Hamilton,1,McLaren,4,2007,4,4,4.0,22.0,3.0,0.0,44.0,11.0,1.0,1
4,40,1,Lewis Hamilton,1,McLaren,6,2007,5,2,2.0,30.0,1.0,0.0,58.0,11.0,1.0,1


In [4]:
modeling_data.isna().sum()

raceId                                  0
driverId                                0
driver_name                             0
constructorId                           0
name_constructor                        0
circuitId                               0
year                                    0
round                                   0
grid                                    0
qualifying_position                 16275
driver_points_before_race               0
driver_standing_before_race             0
driver_wins_before_race                 0
constructor_points_before_race          0
constructor_standing_before_race        0
constructor_wins_before_race            0
podium                                  0
dtype: int64

In [5]:
modeling_data.shape

(26759, 17)

In [6]:
modeling_data = modeling_data.dropna()

modeling_data.shape

(10484, 17)

In [7]:
modeling_data["podium"].value_counts()

podium
0    9002
1    1482
Name: count, dtype: int64

In [8]:
modeling_data["podium"].value_counts(normalize=True)

podium
0    0.858642
1    0.141358
Name: proportion, dtype: float64

## Train, Validation and Test sets

In [9]:
train = modeling_data[modeling_data["year"] <= 2018]

val = modeling_data[
    (modeling_data["year"] >= 2019) &
    (modeling_data["year"] <= 2021)
]

test = modeling_data[modeling_data["year"] >= 2022]

In [10]:
print("Train shape:", train.shape)
print("Validation shape:", val.shape)
print("Test shape:", test.shape)

Train shape: (7928, 17)
Validation shape: (1197, 17)
Test shape: (1359, 17)


### Feature sets

In [11]:
feature_sets = {
    
    "grid_only": [
        "grid"
    ],
    
    "grid_and_qualifying": [
        "grid",
        "qualifying_position"
    ],
    
    "race_context": [
        "grid",
        "qualifying_position",
        "year",
        "round",
        "circuitId"
    ],
    
    "driver_constructor_circuit": [
        "grid",
        "qualifying_position",
        "driverId",
        "constructorId",
        "circuitId"
    ],
    
    "standings_only": [
        "grid",
        "qualifying_position",
        "driver_points_before_race",
        "driver_standing_before_race",
        "driver_wins_before_race",
        "constructor_points_before_race",
        "constructor_standing_before_race",
        "constructor_wins_before_race"
    ],
    
    "all_valid_features": [
        "driverId",
        "constructorId",
        "circuitId",
        "year",
        "round",
        "grid",
        "qualifying_position",
        "driver_points_before_race",
        "driver_standing_before_race",
        "driver_wins_before_race",
        "constructor_points_before_race",
        "constructor_standing_before_race",
        "constructor_wins_before_race"
    ]
}

## Feature Set 1: grid_only
- Logistic Regression
- Random Forest
- Gradient Boosting

In [12]:
features = ["grid"]

X_train = train[features]
Y_train = train["podium"]

X_val = val[features]
Y_val = val["podium"]

In [13]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import GradientBoostingClassifier

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

In [14]:
model_results = []

#### Logistic Regression

In [15]:
lr_model = LogisticRegression(max_iter=1000)

lr_model.fit(X_train, Y_train)

lr_preds = lr_model.predict(X_val)
lr_probs = lr_model.predict_proba(X_val)[:, 1]

model_results.append({
    "feature_set": "grid_only",
    "model": "Logistic Regression",
    "accuracy": accuracy_score(Y_val, lr_preds),
    "precision": precision_score(Y_val, lr_preds, zero_division=0),
    "recall": recall_score(Y_val, lr_preds, zero_division=0),
    "f1": f1_score(Y_val, lr_preds, zero_division=0),
    "roc_auc": roc_auc_score(Y_val, lr_probs)
})

#### Random Forest

In [16]:
rf_model = RandomForestClassifier(random_state=42)

rf_model.fit(X_train, Y_train)

rf_preds = rf_model.predict(X_val)
rf_probs = rf_model.predict_proba(X_val)[:, 1]

model_results.append({
    "feature_set": "grid_only",
    "model": "Random Forest",
    "accuracy": accuracy_score(Y_val, rf_preds),
    "precision": precision_score(Y_val, rf_preds, zero_division=0),
    "recall": recall_score(Y_val, rf_preds, zero_division=0),
    "f1": f1_score(Y_val, rf_preds, zero_division=0),
    "roc_auc": roc_auc_score(Y_val, rf_probs)
})

#### Gradient Boosting

In [17]:
gb_model = GradientBoostingClassifier(random_state=42)

gb_model.fit(X_train, Y_train)

gb_preds = gb_model.predict(X_val)
gb_probs = gb_model.predict_proba(X_val)[:, 1]

model_results.append({
    "feature_set": "grid_only",
    "model": "Gradient Boosting",
    "accuracy": accuracy_score(Y_val, gb_preds),
    "precision": precision_score(Y_val, gb_preds, zero_division=0),
    "recall": recall_score(Y_val, gb_preds, zero_division=0),
    "f1": f1_score(Y_val, gb_preds, zero_division=0),
    "roc_auc": roc_auc_score(Y_val, gb_probs)
})

In [18]:
results_df = pd.DataFrame(model_results)

results_df

,feature_set,model,accuracy,precision,recall,f1,roc_auc
0,grid_only,Logistic Regression,0.878028,0.614865,0.505556,0.554878,0.889405
1,grid_only,Random Forest,0.901420,0.758333,0.505556,0.606667,0.917614
2,grid_only,Gradient Boosting,0.901420,0.758333,0.505556,0.606667,0.918581


Looking the model performance for just starting grid position, we can see that are capturing ~51% of actual podium finishes. In addition the ROC accuracy is around 90%, which makes starting grid position a really strong predictor.

## Feature Set 2: Grid + Qualifying 

- Logistic Regression
- Random Forest
- Gradient Boosting
  
features = ["grid", "qualifying_position"]

In [19]:
features = ["grid", "qualifying_position"]

X_train = train[features]
Y_train = train["podium"]

X_val = val[features]
Y_val = val["podium"]

#### Logistic Regression

In [20]:
lr_model = LogisticRegression(max_iter=1000)

lr_model.fit(X_train, Y_train)

lr_preds = lr_model.predict(X_val)
lr_probs = lr_model.predict_proba(X_val)[:, 1]

model_results.append({
    "feature_set": "grid_qualifying",
    "model": "Logistic Regression",
    "accuracy": accuracy_score(Y_val, lr_preds),
    "precision": precision_score(Y_val, lr_preds, zero_division=0),
    "recall": recall_score(Y_val, lr_preds, zero_division=0),
    "f1": f1_score(Y_val, lr_preds, zero_division=0),
    "roc_auc": roc_auc_score(Y_val, lr_probs)
})

#### Random Foresst

In [21]:
rf_model = RandomForestClassifier(random_state=42)

rf_model.fit(X_train, Y_train)

rf_preds = rf_model.predict(X_val)
rf_probs = rf_model.predict_proba(X_val)[:, 1]

model_results.append({
    "feature_set": "grid_qualifying",
    "model": "Random Forest",
    "accuracy": accuracy_score(Y_val, rf_preds),
    "precision": precision_score(Y_val, rf_preds, zero_division=0),
    "recall": recall_score(Y_val, rf_preds, zero_division=0),
    "f1": f1_score(Y_val, rf_preds, zero_division=0),
    "roc_auc": roc_auc_score(Y_val, rf_probs)
})

#### Gradient Boosting

In [22]:
gb_model = GradientBoostingClassifier(random_state=42)

gb_model.fit(X_train, Y_train)

gb_preds = gb_model.predict(X_val)
gb_probs = gb_model.predict_proba(X_val)[:, 1]

model_results.append({
    "feature_set": "grid_qualifying",
    "model": "Gradient Boosting",
    "accuracy": accuracy_score(Y_val, gb_preds),
    "precision": precision_score(Y_val, gb_preds, zero_division=0),
    "recall": recall_score(Y_val, gb_preds, zero_division=0),
    "f1": f1_score(Y_val, gb_preds, zero_division=0),
    "roc_auc": roc_auc_score(Y_val, gb_probs)
})

In [23]:
results_df = pd.DataFrame(model_results)

results_df.sort_values("roc_auc", ascending=False)

,feature_set,model,accuracy,precision,recall,f1,roc_auc
3,grid_qualifying,Logistic Regression,0.895572,0.735043,0.477778,0.579125,0.921670
4,grid_qualifying,Random Forest,0.908939,0.715152,0.655556,0.684058,0.919496
2,grid_only,Gradient Boosting,0.901420,0.758333,0.505556,0.606667,0.918581
1,grid_only,Random Forest,0.901420,0.758333,0.505556,0.606667,0.917614
5,grid_qualifying,Gradient Boosting,0.907268,0.711656,0.644444,0.676385,0.911958
0,grid_only,Logistic Regression,0.878028,0.614865,0.505556,0.554878,0.889405


## Feature Set 3: Driver History
- Logistic Regression
- Random Forest
- Gradient Boosting

features = [
    "grid",
    "qualifying_position",
    "driver_points_before_race",
    "driver_standing_before_race",
    "driver_wins_before_race"
]

In [24]:
features = [
    "grid",
    "qualifying_position",
    "driver_points_before_race",
    "driver_standing_before_race",
    "driver_wins_before_race"
]

X_train = train[features]
Y_train = train["podium"]

X_val = val[features]
Y_val = val["podium"]

#### Logistic Regression

In [25]:
lr_model = LogisticRegression(max_iter=1000)

lr_model.fit(X_train, Y_train)

lr_preds = lr_model.predict(X_val)
lr_probs = lr_model.predict_proba(X_val)[:, 1]

model_results.append({
    "feature_set": "driver_history",
    "model": "Logistic Regression",
    "accuracy": accuracy_score(Y_val, lr_preds),
    "precision": precision_score(Y_val, lr_preds, zero_division=0),
    "recall": recall_score(Y_val, lr_preds, zero_division=0),
    "f1": f1_score(Y_val, lr_preds, zero_division=0),
    "roc_auc": roc_auc_score(Y_val, lr_probs)
})

#### Random Forest

In [26]:
rf_model = RandomForestClassifier(random_state=42)

rf_model.fit(X_train, Y_train)

rf_preds = rf_model.predict(X_val)
rf_probs = rf_model.predict_proba(X_val)[:, 1]

model_results.append({
    "feature_set": "driver_history",
    "model": "Random Forest",
    "accuracy": accuracy_score(Y_val, rf_preds),
    "precision": precision_score(Y_val, rf_preds, zero_division=0),
    "recall": recall_score(Y_val, rf_preds, zero_division=0),
    "f1": f1_score(Y_val, rf_preds, zero_division=0),
    "roc_auc": roc_auc_score(Y_val, rf_probs)
})

#### Gradient Boosting

In [27]:
gb_model = GradientBoostingClassifier(random_state=42)

gb_model.fit(X_train, Y_train)

gb_preds = gb_model.predict(X_val)
gb_probs = gb_model.predict_proba(X_val)[:, 1]

model_results.append({
    "feature_set": "driver_history",
    "model": "Gradient Boosting",
    "accuracy": accuracy_score(Y_val, gb_preds),
    "precision": precision_score(Y_val, gb_preds, zero_division=0),
    "recall": recall_score(Y_val, gb_preds, zero_division=0),
    "f1": f1_score(Y_val, gb_preds, zero_division=0),
    "roc_auc": roc_auc_score(Y_val, gb_probs)
})

In [28]:
results_df = pd.DataFrame(model_results)

results_df.sort_values("roc_auc", ascending=False)

,feature_set,model,accuracy,precision,recall,f1,roc_auc
8,driver_history,Gradient Boosting,0.923141,0.761905,0.711111,0.735632,0.934207
6,driver_history,Logistic Regression,0.904762,0.732394,0.577778,0.645963,0.925563
3,grid_qualifying,Logistic Regression,0.895572,0.735043,0.477778,0.579125,0.921670
4,grid_qualifying,Random Forest,0.908939,0.715152,0.655556,0.684058,0.919496
2,grid_only,Gradient Boosting,0.901420,0.758333,0.505556,0.606667,0.918581
1,grid_only,Random Forest,0.901420,0.758333,0.505556,0.606667,0.917614
5,grid_qualifying,Gradient Boosting,0.907268,0.711656,0.644444,0.676385,0.911958
7,driver_history,Random Forest,0.893901,0.670968,0.577778,0.620896,0.902753
0,grid_only,Logistic Regression,0.878028,0.614865,0.505556,0.554878,0.889405


## Rest of featurs added to model_results list

In [29]:
remaining_feature_sets = {

    "constructor_history": [
        "grid",
        "qualifying_position",
        "constructor_points_before_race",
        "constructor_standing_before_race",
        "constructor_wins_before_race"
    ],

    "all_valid_features": [
        "driverId",
        "constructorId",
        "circuitId",
        "year",
        "round",
        "grid",
        "qualifying_position",
        "driver_points_before_race",
        "driver_standing_before_race",
        "driver_wins_before_race",
        "constructor_points_before_race",
        "constructor_standing_before_race",
        "constructor_wins_before_race"
    ]
}

In [30]:
models = {

    #fix for scaling issues
    "Logistic Regression": make_pipeline(
        StandardScaler(),
        LogisticRegression(max_iter=5000)
    ),
    
    "Random Forest": RandomForestClassifier(random_state=42),
    "Gradient Boosting": GradientBoostingClassifier(random_state=42)
}

In [31]:
def test_model(model, model_name, feature_set_name, features):
    
    X_train = train[features]
    Y_train = train["podium"]

    X_val = val[features]
    Y_val = val["podium"]

    model.fit(X_train, Y_train)

    preds = model.predict(X_val)
    probs = model.predict_proba(X_val)[:, 1]

    result = {
        "feature_set": feature_set_name,
        "model": model_name,
        "accuracy": accuracy_score(Y_val, preds),
        "precision": precision_score(Y_val, preds, zero_division=0),
        "recall": recall_score(Y_val, preds, zero_division=0),
        "f1": f1_score(Y_val, preds, zero_division=0),
        "roc_auc": roc_auc_score(Y_val, probs)
    }

    return result

In [32]:
for feature_set_name, features in remaining_feature_sets.items():

    for model_name, model in models.items():

        result = test_model(
            model,
            model_name,
            feature_set_name,
            features
        )

        model_results.append(result)


In [33]:
results_df = pd.DataFrame(model_results)

results_df.sort_values("roc_auc", ascending=False)

,feature_set,model,accuracy,precision,recall,f1,roc_auc
8,driver_history,Gradient Boosting,0.923141,0.761905,0.711111,0.735632,0.934207
14,all_valid_features,Gradient Boosting,0.910610,0.735484,0.633333,0.680597,0.931888
11,constructor_history,Gradient Boosting,0.905597,0.721854,0.605556,0.658610,0.926521
12,all_valid_features,Logistic Regression,0.902256,0.752000,0.522222,0.616393,0.925904
6,driver_history,Logistic Regression,0.904762,0.732394,0.577778,0.645963,0.925563
13,all_valid_features,Random Forest,0.904762,0.717105,0.605556,0.656627,0.922269
9,constructor_history,Logistic Regression,0.898079,0.701389,0.561111,0.623457,0.922088
3,grid_qualifying,Logistic Regression,0.895572,0.735043,0.477778,0.579125,0.921670
4,grid_qualifying,Random Forest,0.908939,0.715152,0.655556,0.684058,0.919496
2,grid_only,Gradient Boosting,0.901420,0.758333,0.505556,0.606667,0.918581


Comparing the combination of feature set and the three models, we can see that the best performing model is Gradient Boosting based on the driver_history feature set. This is interesting because it seems like training on all valid features doesn't necessarily mean best performance.  ADD MORE ANALYSIS